In [ ]:
!pip install datasets pillow openai google-genai pandas tqdm

In [ ]:
import os
from datasets import load_dataset
import pandas as pd
from google import genai
from google.genai import types
from PIL import Image
import json
from tqdm import tqdm
import time
import io
import re

# Initialize Gemini client
GOOGLE_API_KEY = ""
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

def extract_dvqa_samples(num_samples=500):
    """
    Extract samples from the DVQA subset of LLaVA-CoT-o1-Instruct dataset
    """
    print("Loading dataset...")
    dataset = load_dataset("5CD-AI/LLaVA-CoT-o1-Instruct", split="train")

    # Filter for DVQA samples
    dvqa_samples = [sample for sample in dataset if sample.get('source') == 'dvqa' or 'dvqa' in str(sample.get('id', '')).lower()]

    # If filtering by source doesn't work, try other identifying features
    if len(dvqa_samples) == 0:
        print("Filtering by alternative methods...")
        dvqa_samples = list(dataset)[:num_samples]

    print(f"Found {len(dvqa_samples)} DVQA samples")

    # Take only the requested number
    selected_samples = dvqa_samples[:min(num_samples, len(dvqa_samples))]

    return selected_samples

def pil_to_gemini_image(image_obj):
    """
    Convert image to format that Gemini accepts
    Handles both PIL Images and Gemini-generated images
    """
    # If it's already a Gemini Part, return it
    if isinstance(image_obj, types.Part):
        return image_obj

    # Try to handle as PIL Image
    try:
        img_byte_arr = io.BytesIO()

        # Check if it's a standard PIL Image
        if hasattr(image_obj, 'save') and hasattr(image_obj, 'format'):
            # Standard PIL Image
            image_obj.save(img_byte_arr, format='PNG')
        else:
            # Gemini generated image - save differently
            import tempfile
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
                image_obj.save(tmp.name)
                with open(tmp.name, 'rb') as f:
                    img_byte_arr.write(f.read())
                os.unlink(tmp.name)

        img_byte_arr.seek(0)

        # Create a Part with inline data
        image_part = types.Part.from_bytes(
            data=img_byte_arr.read(),
            mime_type="image/png"
        )

        return image_part

    except Exception as e:
        print(f"Error converting image: {e}")
        # If all else fails, try to read from the saved file
        if hasattr(image_obj, 'filename'):
            with open(image_obj.filename, 'rb') as f:
                image_part = types.Part.from_bytes(
                    data=f.read(),
                    mime_type="image/png"
                )
            return image_part
        raise

def judge_arabic_translation(arabic_text, gemini_client):
    """
    Use LLM as a judge to verify the translation is pure Arabic without English words
    Returns: (is_valid: bool, english_words_found: list)
    """
    prompt = f"""أنت حكم لغوي متخصص. مهمتك هي فحص النص التالي والتأكد من أنه مكتوب بالعربية فقط بدون أي كلمات إنجليزية.

النص المراد فحصه:
{arabic_text}

قم بما يلي:
1. افحص النص بعناية
2. ابحث عن أي كلمات أو حروف إنجليزية
3. الأرقام مقبولة (0-9)
4. أعد إجابتك بهذا التنسيق فقط:

VALID: [نعم أو لا]
ENGLISH_WORDS: [اذكر الكلمات الإنجليزية إن وجدت، أو "لا يوجد"]

مثال إذا كان النص صحيح:
VALID: نعم
ENGLISH_WORDS: لا يوجد

مثال إذا كان النص يحتوي على إنجليزية:
VALID: لا
ENGLISH_WORDS: hello, world, test"""

    try:
        response = gemini_client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=prompt,
        )

        result = response.text.strip()

        # Parse the response
        is_valid = "نعم" in result.split("VALID:")[-1].split("\n")[0] if "VALID:" in result else False

        english_words = []
        if "ENGLISH_WORDS:" in result:
            words_line = result.split("ENGLISH_WORDS:")[-1].strip()
            if "لا يوجد" not in words_line and words_line:
                english_words = [w.strip() for w in words_line.split(",")]

        return is_valid, english_words

    except Exception as e:
        print(f"Judge error: {e}")
        # If judge fails, do a simple regex check
        english_pattern = re.compile(r'[a-zA-Z]{2,}')  # 2+ consecutive English letters
        english_found = english_pattern.findall(arabic_text)
        return len(english_found) == 0, english_found

def generate_arabic_image_with_gemini(original_image, gemini_client):
    """
    Generate a new image with Arabic text using Google Gemini
    """
    prompt = """انظر إلى هذه الصورة بعناية. هذه صورة تحتوي على رسم بياني أو مخطط بنصوص إنجليزية.

أريدك أن تنشئ صورة جديدة مماثلة تماماً لهذه الصورة، ولكن مع تغيير جميع النصوص الإنجليزية إلى نصوص عربية.

تفاصيل مهمة:
1. احتفظ بنفس نوع الرسم البياني (bar chart, line graph, pie chart, etc.)
2. استخرج جميع النصوص الإنجليزية الموجودة (العناوين، التسميات، أسماء المحاور، القيم، الأرقام)
3. ترجم كل نص إلى العربية
4. أنشئ صورة جديدة تحتوي على:
   - نفس البيانات والقيم بالضبط
   - نفس الألوان
   - نفس التخطيط والبنية
   - جميع النصوص بالعربية فقط
   - النصوص العربية مكتوبة من اليمين إلى اليسار
   - نفس حجم ونسب الصورة الأصلية

أنشئ الصورة الآن."""

    try:
        response = gemini_client.models.generate_content(
            model="gemini-3-pro-image-preview",
            contents=[prompt, original_image],
        )

        # Extract generated image from response
        generated_image = None
        for part in response.parts:
            if part.inline_data is not None:
                generated_image = part.as_image()
                break

        if generated_image:
            print("✓ Arabic image generated successfully")
            return generated_image
        else:
            print("✗ No image found in response")
            return None

    except Exception as e:
        print(f"Gemini image generation error: {e}")
        import traceback
        traceback.print_exc()
        return None

def translate_question_with_image_gemini(original_image, english_question, gemini_client, max_retries=3):
    """
    Translate question to Arabic WITH the image to ensure consistent terminology
    Uses LLM judge to verify pure Arabic translation
    """
    if not english_question or english_question.strip() == "":
        print("  Warning: Empty question provided for translation")
        return ""

    for attempt in range(max_retries):
        prompt = f"""انظر إلى هذه الصورة واقرأ السؤال الإنجليزي التالي.

السؤال الإنجليزي: {english_question}

مهمتك:
1. انظر إلى المصطلحات والتسميات الموجودة في الصورة وكيف تمت ترجمتها إلى العربية
2. ترجم السؤال الكامل إلى اللغة العربية بدقة - احتفظ بكل الأجزاء بما في ذلك المقدمات والتعليمات
3. ترجم المعنى وليس مجرد الكلمات
4. استخدم نفس المصطلحات العربية الموجودة في الصورة إذا كانت متعلقة بالسؤال
5. تجنب الترجمة الصوتية (transliteration) - استخدم الترجمة المعنوية (semantic translation)
6. احتفظ بنفس البنية والتنسيق (إذا كان هناك "Question:" احتفظ به كـ "السؤال:")
7. أعد الترجمة العربية الكاملة بدون حذف أي أجزاء
8. مهم جداً: يجب أن تكون الترجمة بالعربية فقط، بدون أي كلمات إنجليزية أو ترجمة صوتية

"""

        try:
            response = gemini_client.models.generate_content(
                model="gemini-3-flash-preview",
                contents=[prompt, original_image],
            )

            translation = response.text.strip()

            # Remove any extra text that Gemini might add
            if "الترجمة العربية:" in translation:
                translation = translation.split("الترجمة العربية:")[-1].strip()

            # Judge the translation
            print(f"  Judging translation (attempt {attempt + 1}/{max_retries})...")
            is_valid, english_words = judge_arabic_translation(translation, gemini_client)

            if is_valid:
                print(f"  ✓ Translation validated: Pure Arabic")
                return translation
            else:
                print(f"  ✗ English words found: {english_words}")
                if attempt < max_retries - 1:
                    print(f"  Retrying translation...")
                    time.sleep(1)
                else:
                    print(f"  Max retries reached, using best attempt")
                    return translation

        except Exception as e:
            print(f"Translation error: {e}")
            import traceback
            traceback.print_exc()
            if attempt == max_retries - 1:
                return english_question

    return english_question

def load_existing_ids_jsonl(path: str) -> set:
    ids = set()
    if not os.path.exists(path):
        return ids
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if "id" in obj:
                    ids.add(str(obj["id"]))
            except json.JSONDecodeError:
                continue
    return ids

import re, glob

def process_dataset(samples, gemini_client, output_file="arabic_dvqa_dataset.jsonl", images_dir="arabic_images"):
    os.makedirs(images_dir, exist_ok=True)

    existing_ids = load_existing_ids_jsonl(output_file)
    print(f"Already in {output_file}: {len(existing_ids)}")

    existing_imgs = glob.glob(os.path.join(images_dir, "arabic_image_*.png"))
    if existing_imgs:
        nums = []
        for p in existing_imgs:
            m = re.search(r"arabic_image_(\d+)\.png$", p)
            if m:
                nums.append(int(m.group(1)))
        start_index = (max(nums) + 1) if nums else 0
    else:
        start_index = 421

    print("Starting image index from:", start_index)

    results = []
    local_idx = 0  # counts ONLY new processed samples

    for _, sample in enumerate(tqdm(samples, desc="Processing samples")):
        arabic_image_path = None
        try:
            sample_id = str(sample.get("id", ""))
            if not sample_id:
                continue

            # ✅ Skip if already processed
            if sample_id in existing_ids:
                continue

            image = sample.get("image")

            # question
            question = sample["question"]


            # Step 1: Generate Arabic image
            arabic_image = generate_arabic_image_with_gemini(image, gemini_client)
            if arabic_image is None:
                continue

            # ✅ numbered filename
            img_idx = start_index + local_idx
            arabic_image_path = os.path.join(images_dir, f"arabic_image_{img_idx}.png")
            arabic_image.save(arabic_image_path)

            # Step 2: Translate question
            arabic_question = translate_question_with_image_gemini(image, question_clean, gemini_client)
            if not arabic_question:
                if arabic_image_path and os.path.exists(arabic_image_path):
                    os.remove(arabic_image_path)
                continue

            if not arabic_output:
                if arabic_image_path and os.path.exists(arabic_image_path):
                    os.remove(arabic_image_path)
                continue

            result = {
                "id": sample_id,
                "image": arabic_image_path,
                "question": arabic_question,
                "ground_truth": sample.get("ground_truth"),
            }

            with open(output_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            results.append(result)
            existing_ids.add(sample_id)
            local_idx += 1  # ✅ increment only when saved

            time.sleep(3)

        except Exception:
            try:
                if arabic_image_path and os.path.exists(arabic_image_path):
                    os.remove(arabic_image_path)
            except:
                pass
            continue

    return results



def main():

    # Initialize Gemini client
    gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

    # Extract 500 DVQA samples
    print("=" * 60)
    print("DVQA to Arabic Dataset Processing Pipeline")
    print("Using Google Gemini with LLM Judge")
    print("=" * 60)
    print("\nModels used:")
    print("  - Image generation: gemini-2.5-flash-image")
    print("  - Text/Vision: gemini-3-flash")
    print("=" * 60)
    print("\nExtracting DVQA samples from dataset...")
    samples = extract_dvqa_samples(num_samples=1666)

    results = process_dataset(samples, gemini_client)

    # Save complete results
    with open("arabic_dvqa_dataset_complete.json", 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


    print("\n" + "=" * 60)
    print(f"✓ Processing complete! Generated {len(results)} Arabic samples.")

if __name__ == "__main__":
    main()


In [ ]:
from google.colab import files
!zip -r arabic_images.zip arabic_images
files.download("arabic_images.zip")


  adding: arabic_images/ (stored 0%)
  adding: arabic_images/arabic_image_458.png (deflated 2%)
  adding: arabic_images/arabic_image_433.png (deflated 4%)
  adding: arabic_images/arabic_image_464.png (deflated 3%)
  adding: arabic_images/arabic_image_422.png (deflated 4%)
  adding: arabic_images/arabic_image_435.png (deflated 4%)
  adding: arabic_images/arabic_image_446.png (deflated 4%)
  adding: arabic_images/arabic_image_462.png (deflated 2%)
  adding: arabic_images/arabic_image_469.png (deflated 4%)
  adding: arabic_images/arabic_image_473.png (deflated 4%)
  adding: arabic_images/arabic_image_459.png (deflated 3%)
  adding: arabic_images/arabic_image_442.png (deflated 3%)
  adding: arabic_images/arabic_image_438.png (deflated 4%)
  adding: arabic_images/arabic_image_476.png (deflated 4%)
  adding: arabic_images/arabic_image_450.png (deflated 3%)
  adding: arabic_images/arabic_image_453.png (deflated 3%)
  adding: arabic_images/arabic_image_449.png (deflated 3%)
  adding: arabic_im

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>